# Drew's Test Code

## 1. Open data

In [ ]:
import pandas as pd
import geopandas as gpd
import dask_geopandas as dgpd
import matplotlib.pyplot as plt

In [ ]:
USA_48_STATES_BBOX = (-128.320313,24.367114,-65.039063,50.401515)

In [ ]:
us_chargers = pd.read_csv('data/USChargingLocations.csv')
us_chargers = gpd.GeoDataFrame(us_chargers, geometry = gpd.points_from_xy(us_chargers['Longitude'], us_chargers['Latitude']))
us_chargers = us_chargers.cx[USA_48_STATES_BBOX[0]:USA_48_STATES_BBOX[2], USA_48_STATES_BBOX[1]:USA_48_STATES_BBOX[3]]

In [ ]:
interstates = gpd.read_file('data/spatial_interstates_w_mobility.geojson', bbox = USA_48_STATES_BBOX)
interstates.shape

In [ ]:
ax = interstates.plot(color='grey')
us_chargers.plot(ax = ax, color='red', markersize=0.5)
ax.set_axis_off()

In [ ]:
us_chargers = us_chargers.set_crs('EPSG:4326')
us_chargers.crs == interstates.crs

## 2. Filter US Chargers

In [ ]:
interstates_1_mile = interstates.to_crs('EPSG:9311').buffer(1609.344)
# merged_interstates_buffer = interstates_1_mile.union_all()

In [ ]:
chargers_by_interstate = gpd.sjoin(
    us_chargers.to_crs('EPSG:9311'), 
    gpd.GeoDataFrame(geometry=interstates_1_mile), 
    how="inner", 
    predicate="intersects"
).drop_duplicates(subset='ID')

In [ ]:
# dask_chargers = dgpd.from_geopandas(us_chargers.to_crs('EPSG:9311'), npartitions=4)
# result = dask_chargers[dask_chargers.within(merged_interstates_buffer)]
# chargers_near_interstate = result.compute()

In [ ]:
# chargers_near_interstate = us_chargers.to_crs('EPSG:9311').within(merged_interstates_buffer)

In [ ]:
interstates_simple = interstates.to_crs('EPSG:9311')
interstates_simple['geometry'] = interstates_simple.simplify(500)
interstates_simple = interstates_simple[interstates_simple.is_valid & ~interstates_simple.is_empty]

In [ ]:
print(interstates_simple.total_bounds)

In [ ]:
fig, ax = plt.subplots(figsize=(15, 10))

charger_buffers = chargers_by_interstate.to_crs('EPSG:9311')
charger_buffers = charger_buffers.buffer(1609.344 * 25)

charger_buffers = charger_buffers[charger_buffers.is_valid & ~charger_buffers.is_empty]

print("Interstate Bounds:", interstates_simple.total_bounds)
print("Buffer Bounds:", charger_buffers.total_bounds)

interstates_simple.plot(ax = ax,
                        alpha = 0.5, 
                        column='aadt', 
                        cmap='viridis',
                        legend=False)

charger_buffers.plot(ax = ax, color='red', markersize=0.5)

ax.set_xlim(-2500000, 2500000)
ax.set_ylim(-1500000, 1500000)

ax.set_aspect('auto')

ax.set_axis_off()

In [ ]:
interstates.columns